# Hit callers on cells and on guide profiles

{func}`~mantispy.tl.hit_calling` offers three statistics, and a screen can be tested at two stages: on the single
cells, or on one median profile per guide. This page runs every method that applies at each stage on the same screen,
and compares the calls with a set of genes whose answer is known.

The screen is {func}`~mantispy.ds.scallops_arv471`, an optical pooled CRISPR screen in MCF7 cells treated with the
estrogen-receptor degrader ARV-471. The drug needs the CRL4-CRBN ligase to degrade the receptor, so knocking out
`CRBN`, `DDB1`, `CUL4A`, `CUL4B` or the receptor itself (`ESR1`) changes the phenotype: these five are the known
positives. Twelve olfactory-receptor genes are not expressed in these cells and are the known negatives. The other
genes of the library have no known answer, so they are counted but not scored.

In [1]:
import time

import pandas as pd

import mantispy as mt

POSITIVES = ["CRBN", "DDB1", "CUL4A", "CUL4B", "ESR1"]

cells = mt.ds.scallops_arv471()
mt.pp.normalize(cells, by="Metadata_Plate", reference="negcon")
genes = cells.obs["Metadata_Gene"].astype(str)
NEGATIVES = sorted(gene for gene in genes.unique() if gene.startswith("OR"))
# Two million cells is more than the comparison needs; keep 200 per guide.
cells = mt.pp.downsample(cells, n_per_group=200, by="Metadata_sgRNA")
cells

AnnData object with n_obs × n_vars = 135549 × 9
    obs: 'Metadata_Gene', 'Metadata_sgRNA', 'Metadata_Control_Type', 'Metadata_Control', 'Metadata_Perturbation', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Row', 'Metadata_Col', 'Metadata_ImageID', 'Metadata_ObjectType', 'Metadata_ObjectNumber', 'Metadata_Perturbation_Type'
    var: 'object', 'feature_group', 'feature', 'channel', 'scale', 'angle', 'gray_levels', 'radial_bin', 'params', 'is_feature', 'feature_kind', 'degenerate_scale'
    uns: 'mantispy'
    layers: None (.X)

In [2]:
def score(calls):
    """Confusion counts of each method's calls against the known positives and negatives."""
    rows = {}
    for method, called in calls.items():
        rows[method] = {
            "positives called": int(called.reindex(POSITIVES).sum()),
            "positives missed": int((~called.reindex(POSITIVES)).sum()),
            "negatives called": int(called.reindex(NEGATIVES).sum()),
            "negatives not called": int((~called.reindex(NEGATIVES)).sum()),
            "other genes called": int(called.drop(POSITIVES + NEGATIVES + ["nontargeting"], errors="ignore").sum()),
        }
    return pd.DataFrame(rows).T


def agreement(calls):
    """Share of genes on which two methods make the same call."""
    frame = pd.DataFrame(calls)
    return pd.DataFrame({a: {b: (frame[a] == frame[b]).mean() for b in frame} for a in frame}).round(2)


def call(adata, methods):
    """Run each method on the genes and return its calls at q < 0.05 and its runtime."""
    calls, seconds = {}, {}
    for name, keywords in methods.items():
        start = time.perf_counter()
        mt.tl.hit_calling(adata, groupby="Metadata_Gene", n_permutations=1000, key_added=name, **keywords)
        seconds[name] = round(time.perf_counter() - start, 1)
        calls[name] = adata.uns["mantispy"][name].set_index("group")["qvalue"] < 0.05
    return calls, pd.Series(seconds, name="seconds")

## Single cells

Every well of a pooled screen holds every guide, so the unit the null draws whole is the guide (`replicate=`).
The three statistics ask different questions of a gene's cells:

- `"mahalanobis"`: how far the gene's typical cell sits from the non-targeting cells, in the controls' covariance.
- `"ks"`: whether the distribution of the cells' distances differs, which also catches a subpopulation that moved.
- `"hellinger"`: how little a Gaussian fitted to the gene's cells overlaps one fitted to the controls, so a change in
  spread or shape counts as well as a shift. It follows the robust morphological perturbation value of BioProfiling.jl
  (Vulliard et al., *Bioinformatics* 2022, [doi:10.1093/bioinformatics/btab853](https://doi.org/10.1093/bioinformatics/btab853)),
  which builds on the multidimensional perturbation value (Hutz et al., *J Biomol Screen* 2013,
  [doi:10.1177/1087057112469257](https://doi.org/10.1177/1087057112469257)).
  `covariance="robust"` gives BioProfiling.jl's robust variant; it refits the fits on every permutation and takes far
  longer, so it is left out here.

In [3]:
cell_calls, cell_seconds = call(
    cells,
    {
        "mahalanobis": {"replicate": "Metadata_sgRNA"},
        "ks": {"method": "ks", "replicate": "Metadata_sgRNA"},
        "hellinger": {"method": "hellinger", "replicate": "Metadata_sgRNA"},
    },
)
score(cell_calls).assign(seconds=cell_seconds)

,positives called,positives missed,negatives called,negatives not called,other genes called,seconds
mahalanobis,2,3,1,11,46,6.8
ks,2,3,1,11,55,21.7
hellinger,2,3,0,12,5,60.8


In [4]:
agreement(cell_calls)

,mahalanobis,ks,hellinger
mahalanobis,1.00,0.91,0.74
ks,0.91,1.00,0.68
hellinger,0.74,0.68,1.00


All three methods call two of the five positives and miss the other three. `"mahalanobis"` and `"ks"` each call one
of the twelve negatives, and `"hellinger"` calls none. The methods differ most among the genes without a known answer:
`"hellinger"` calls 5 of them, `"mahalanobis"` 46 and `"ks"` 55. The truth set cannot say which of those calls are
right, but it shows that `"hellinger"` returns the shortest list at the same recall on the known positives. It is also
the slowest of the three. `"mahalanobis"` and `"ks"` agree on 91% of the genes, and `"hellinger"` agrees with them on
74% and 68%.

## Guide profiles

The same normalized cells, collapsed to one median profile per guide with {func}`~mantispy.tl.aggregate`, so that the
two stages see the same measurements. A gene now has a handful of rows, its guides. `"hellinger"` needs a covariance
within each group, and a covariance in nine features needs at least twice as many rows as features on each side, so it
does not apply at this stage; the other two do, and each guide profile is its own unit.

In [5]:
guides = mt.tl.aggregate(cells, by=("Metadata_Gene", "Metadata_sgRNA"), func="median")
guide_calls, guide_seconds = call(guides, {"mahalanobis": {}, "ks": {"method": "ks"}})
score(guide_calls).assign(seconds=guide_seconds)

,positives called,positives missed,negatives called,negatives not called,other genes called,seconds
mahalanobis,1,4,0,12,3,0.1
ks,2,3,0,12,11,0.1


In [6]:
agreement(guide_calls)

,mahalanobis,ks
mahalanobis,1.00,0.93
ks,0.93,1.00


On guide profiles neither method calls a negative. `"mahalanobis"` finds one positive and `"ks"` two, along with 3
and 11 other genes, and each runs in a fraction of a second because a gene now has four rows instead of eight hundred.

## Cells or profiles

On this screen `"mahalanobis"` recovers more positives from the cells than from the guide medians, and `"ks"` recovers
two at both stages. Guide profiles are quick to test and need nothing beyond {func}`~mantispy.tl.aggregate`.
Cells take longer, but they are the only stage where `"hellinger"` applies and where a change in the spread of a gene's
cells can show at all. `"mahalanobis"` stays the default because here it finds as many positives as either alternative
at a fraction of their runtime. `"hellinger"` is the choice when a shorter list, built on the shape of the cell cloud
as well as its centre, matters more than speed.